# Task

Here is the cleaned text with corrected punctuation and clearer structure:

**Extract the algorithm from your Stata/R/Matlab code:**

1. **Input**
2. **Steps**
3. **Output**
4. **Tests**

Then ask an **LLM** to produce a **Python implementation**.
After that, try to **attack or stress-test the implementation with additional tests** (you could even ask the LLM itself to generate these tests).


# Solution

I have chosen the following short Stata code for my HA:

// clear all cookies
clear all

//To make Stata continue the output instead of saying -more- when output exceeds one screen-full
set more off

// upload the data and set the working directory.
global Hosni_path C:\Users\Popov\Documents\Research\Hosni\Family_economics\CPS_new2
global My_path C:\Users\Popov\Documents\Research\Hosni\Family_economics\Project

// cd "$Hosni_path"
cd "$My_path"
use "Time_use"

***** General Summary
// get the labels of each variables (delete "qui" to see results)
qui label list

// see all unique values of the variable "state"
qui levelsof state

*Saving summary statistics in latex using "estout":
eststo clear
estpost summarize  
esttab using general_summary_time_use.tex, cells("mean(fmt(3)) sd(fmt(3)) min(fmt(3)) max(fmt(3))") ///
 label title({Summary Statistics, time use data}) nonumber replace

**********  see how many NaNs are in each variable

// Loop through each variable in the dataset
foreach var of varlist * {
    // Count the total number of observations
    local total = _N

    // Count the number of missing values for the variable
    qui count if missing(`var')
    local missing = r(N)

    // Display the result
    di "`var' has " `missing' " missing values out of " `total'
} // actually, many missed values

So, the algorithm is roughly described by my comments. SO, I use the following text file as prompt:

clear all cookies

To make Stata continue the output instead of saying -more- when output exceeds one screen-full

upload the data and set the working directory.

***** General Summary

get the labels of each variables (delete "qui" to see results)

see all unique values of the variable "state"

Saving summary statistics in latex using "estout":

see how many NaNs are in each variable

Loop through each variable in the dataset

Count the total number of observations

Count the number of missing values for the variable

Display the result

actually, many missed values

The overall prompt to Chat GPT is: 

I need you to help me to implement my Stata code in python. Here are the comments, which roughly describe my code. I want you to give me python code based on these comments. clear all cookies

To make Stata continue the output instead of saying -more- when output exceeds one screen-full

upload the data and set the working directory.

***** General Summary

get the labels of each variables (delete "qui" to see results)

see all unique values of the variable "state"

Saving summary statistics in latex using "estout":

see how many NaNs are in each variable

Loop through each variable in the dataset

Count the total number of observations

Count the number of missing values for the variable

Display the result

actually, many missed values

In [3]:
# !pip install pandas pyreadstat

In [4]:
# GPT code:
import os
import pandas as pd
import pyreadstat

# =========================================
# 1. Set working directory and load data
# =========================================

hosni_path = r"C:\Users\Popov\Documents\Research\Hosni\Family_economics\CPS_new2"
my_path = r"C:\Users\Popov\Documents\Studies\IMT_studies\Python\Into_python_1\HA\HA1"

# choose working directory
os.chdir(my_path)

# load Stata dataset
df, meta = pyreadstat.read_dta("Time_use.dta")

# =========================================
# 2. General summary
# =========================================

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

# =========================================
# 3. Variable labels
# =========================================

print("\nVariable labels:")
for var in df.columns:
    label = meta.column_labels[meta.column_names.index(var)] if var in meta.column_names else None
    print(f"{var}: {label}")

# =========================================
# 4. Unique values of the variable 'state'
# =========================================

if "state" in df.columns:
    print("\nUnique values of 'state':")
    print(sorted(df["state"].dropna().unique()))
else:
    print("\nVariable 'state' not found in dataset.")

# =========================================
# 5. Summary statistics
#    analogous to estpost summarize / esttab
# =========================================

summary = df.describe(include="all").T

# keep only columns similar to Stata output where possible
summary_table = pd.DataFrame(index=df.columns)
summary_table["mean"] = df.mean(numeric_only=True)
summary_table["sd"] = df.std(numeric_only=True)
summary_table["min"] = df.min(numeric_only=True, skipna=True)
summary_table["max"] = df.max(numeric_only=True, skipna=True)

print("\nSummary statistics:")
print(summary_table)

# save to LaTeX
summary_table.to_latex(
    "general_summary_time_use.tex",
    float_format="%.3f",
    na_rep="",
    caption="Summary Statistics, time use data",
    label="tab:summary_time_use"
)

# =========================================
# 6. Count missing values in each variable
# =========================================

print("\nMissing values by variable:")
total = len(df)

for var in df.columns:
    missing = df[var].isna().sum()
    print(f"{var} has {missing} missing values out of {total}")

# =========================================
# 7. Optional: save missing-value report
# =========================================

missing_report = pd.DataFrame({
    "variable": df.columns,
    "missing_count": [df[col].isna().sum() for col in df.columns],
    "total_obs": total
})

missing_report["missing_share"] = missing_report["missing_count"] / missing_report["total_obs"]

missing_report = missing_report.sort_values("missing_count", ascending=False)

print("\nVariables with most missing values:")
print(missing_report)

missing_report.to_csv("missing_report_time_use.csv", index=False)

Dataset shape: (169958, 49)

First 5 rows:
   sample  pernum    ident  hhid  pid  serial  year age civstat cohab  ...  \
0    1930       1  10101.0   1.0  1.0       1  1931  42       1     0  ...   
1    1930       1  10102.0   1.0  1.0       2  1931  42       1     0  ...   
2    1930       1  10103.0   1.0  1.0       3  1931  42       1     0  ...   
3    1930       1  10104.0   1.0  1.0       4  1931  42       1     0  ...   
4    1930       1  10105.0   1.0  1.0       5  1931  42       1     0  ...   

  act_inhome  act_media act_missing act_outhome act_pcare act_physical  \
0          5         90           0           0       620           10   
1        105        110           0           0       615           85   
2         75        120           0         145       635            5   
3          0        335           0           0       655           10   
4         40        115           0           0       655           45   

  act_travel act_undom act_work female  
0 

Testing

In [6]:
# From Stata
data = [
("sample",0,169958),
("pernum",0,169958),
("ident",0,169958),
("hhid",146553,169958),
("pid",0,169958),
("serial",0,169958),
("year",0,169958),
("age",833,169958),
("civstat",9483,169958),
("cohab",21754,169958),
("educ",2189,169958),
("recwght",525,169958),
("ethnic",6098,169958),
("hisp",12522,169958),
("ageyngst",81434,169958),
("ownhome",16798,169958),
("state",4608,169958),
("regione",6998,169958),
("hhtype",7504,169958),
("famstat",2808,169958),
("nadult",630,169958),
("under5",9951,169958),
("under18",708,169958),
("empstat",157,169958),
("incomeqt",24893,169958),
("empsp",83965,169958),
("wagelm",85573,169958),
("wkhrs",67222,169958),
("student",5028,169958),
("occup",18116,169958),
("homemakr",7371,169958),
("retired",13719,169958),
("nwork",18045,169958),
("nchild",169433,169958),
("agekid",169622,169958),
("hhldsize",169433,169958),
("act_chcare",0,169958),
("act_civic",0,169958),
("act_educa",0,169958),
("act_inhome",0,169958),
("act_media",0,169958),
("act_missing",0,169958),
("act_outhome",0,169958),
("act_pcare",0,169958),
("act_physical",0,169958),
("act_travel",0,169958),
("act_undom",0,169958),
("act_work",0,169958),
("female",4,169958)
]

df_missing = pd.DataFrame(data, columns=["variable","missing_values","total_observations"])

print(df_missing)

        variable  missing_values  total_observations
0         sample               0              169958
1         pernum               0              169958
2          ident               0              169958
3           hhid          146553              169958
4            pid               0              169958
5         serial               0              169958
6           year               0              169958
7            age             833              169958
8        civstat            9483              169958
9          cohab           21754              169958
10          educ            2189              169958
11       recwght             525              169958
12        ethnic            6098              169958
13          hisp           12522              169958
14      ageyngst           81434              169958
15       ownhome           16798              169958
16         state            4608              169958
17       regione            6998              

In [7]:
# merge the two reports
check = missing_report.merge(
    df_missing,
    on="variable",
    how="outer",
    suffixes=("_python", "_stata")
)

# compare missing counts
check["match_missing"] = (
    check["missing_count"] == check["missing_values"]
)

# compare total observations (optional)
check["match_total"] = (
    check["total_obs"] == check["total_observations"]
)

# overall check
check["match_all"] = check["match_missing"] & check["match_total"]

print(check)

        variable  missing_count  total_obs  missing_share  missing_values  \
0     act_chcare              0     169958       0.000000               0   
1      act_civic              0     169958       0.000000               0   
2      act_educa              0     169958       0.000000               0   
3     act_inhome              0     169958       0.000000               0   
4      act_media              0     169958       0.000000               0   
5    act_missing              0     169958       0.000000               0   
6    act_outhome              0     169958       0.000000               0   
7      act_pcare              0     169958       0.000000               0   
8   act_physical              0     169958       0.000000               0   
9     act_travel              0     169958       0.000000               0   
10     act_undom              0     169958       0.000000               0   
11      act_work              0     169958       0.000000               0   

In [8]:
# Only mismatched
check_mismatch = check[~check["match_all"]]
print(check_mismatch)

Empty DataFrame
Columns: [variable, missing_count, total_obs, missing_share, missing_values, total_observations, match_missing, match_total, match_all]
Index: []


So, test is passed.